<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_101_tokenizer_analysis_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🔤 **Module 101 — Tokenizer Analysis Workshop** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 15 · Mechanistic Interpretability Workshop (Cohen ML4LLM deep-dives)


# Module 101 — Tokenizer Analysis Workshop

> M55 + M63 built tokenizers from scratch (BPE). This module **opens up trained tokenizers as data** — the hands-on companion to Cohen's "50 ML projects to understand LLMs" **Chapter 2** (projects 1-7). You'll treat tokens as a statistical object: distribution of token lengths, **compression ratio** vs raw bytes, language-fairness in multilingual tokenization, frequency tables, and how three different tokenizer families (**BPE · WordPiece · SentencePiece-Unigram**) split the same text differently. By the end you'll know how to *measure* a tokenizer's quality and bias before you ever fine-tune on top of it.

### What you'll cover
1. The **three tokenization schemes** in production: BPE · WordPiece · Unigram
2. **Book lengths in characters, words, tokens** — choosing the right unit
3. **Pandas frequency tables** of token lengths
4. **Token lengths in characters and bytes** — multilingual headroom
5. **Is tokenization compression?** — entropy and BPC
6. **Multilingual fairness** — why English costs less per character than Tamil or Korean
7. **Vocabulary collisions** — how rare tokens behave at sampling time
8. **Tokenizer fingerprinting** — identifying a model from its splits alone
9. **Practical pitfalls** — leading-space tokens, byte-fallback, ChatML special tokens
10. The **2026 picker** — what to use for a new fine-tune


## 1 · BPE · WordPiece · Unigram — three schemes

| Scheme | Algorithm | Used by |
|---|---|---|
| **BPE** (Sennrich 2016 → tiktoken) | Greedy pair-merge: start with bytes, repeatedly merge the most-frequent adjacent pair until vocab size hit | **GPT-2/3/4o · Llama · DeepSeek · Mistral · Qwen · Gemini-Nano** (byte-level BPE) |
| **WordPiece** (Schuster 2012 → BERT) | Greedy merge maximizing training-data likelihood (vs frequency) | BERT, DistilBERT, MobileBERT |
| **SentencePiece Unigram** (Kudo 2018) | Start with huge vocab, iteratively drop tokens that hurt likelihood least | T5, mT5, ALBERT, XLNet, PaLM |
| **Tiktoken** (cl100k, o200k) | Production-optimized BPE at byte level | GPT-3.5+, GPT-4, OpenAI embed |

**The fundamental contract.** Every tokenizer maps a string `s ∈ Σ*` to a sequence of integers `(t_1, t_2, ..., t_n) ∈ V^n`. Differences:
- **BPE / Tiktoken** is deterministic and greedy — fast, no ambiguity.
- **WordPiece** is deterministic but uses *likelihood* under a unigram model — slightly better lexical fit.
- **Unigram** is *probabilistic* — multiple valid segmentations of the same string; choose argmax or sample.

> 🧠 **Why the choice ripples downstream.** A model trained with one tokenizer cannot be served with another. Switching tokenizers requires retraining from scratch (or a costly vocabulary-transfer recipe, e.g. **Tokadapt** 2024). This is why every model card lists its tokenizer prominently — it's part of the model's identity.


In [ ]:
# Three tokenizers, same string. Cohen Ch2 proj 1.
from transformers import AutoTokenizer

text = "Tokenization is the gateway drug to NLP."
tokenizers = {
    "BPE (GPT-2)":        AutoTokenizer.from_pretrained("gpt2"),
    "WordPiece (BERT)":   AutoTokenizer.from_pretrained("bert-base-uncased"),
    "Unigram (T5)":       AutoTokenizer.from_pretrained("t5-base"),
}
for name, tok in tokenizers.items():
    pieces = tok.tokenize(text)
    ids    = tok.encode(text, add_special_tokens=False)
    print(f"{name:>22}: {len(pieces):>2} tokens — {pieces}")


## 2 · Book lengths — characters, words, tokens

UDL Cohen proj 2 takes a few public-domain books (Pride & Prejudice, Moby Dick, Dracula) and compares:

| Unit | What it measures |
|---|---|
| **Characters** | True string length; the substrate. Locale/encoding-stable |
| **Words** (whitespace-split) | Linguistic unit; varies by language (Chinese has no spaces) |
| **Tokens** | Model-internal unit; what you actually pay for at API time |

A typical English book:
```
characters : ~600,000
words      : ~110,000
tokens (BPE-GPT-2): ~135,000
ratio chars/tokens ≈ 4.4    ← English's "tokens-per-character" cost
```

For an LLM API priced at $0.50 / 1M output tokens (May 2026 Claude-3.5-Haiku rate):
```
$ to print this book = (135 000 / 1 000 000) × $0.50 = $0.067
```

That's the actual *unit economics* — tokens *not* characters. Knowing the conversion factor for your domain is the difference between a $1k and $50k monthly API bill at scale.


## 3 · Frequency tables — Pandas as a tokenizer microscope

Cohen proj 3 builds a frequency histogram of token lengths (measured in characters per token):

```
token-length-in-chars   count    pct
1                       8421     23.4%
2                       7812     21.7%
3                       6109     17.0%
4                       5004     13.9%
5                       3201      8.9%
6                       2105      5.8%
7+                      3309      9.2%
```

Two findings repeat across BPEs:
1. **~50% of tokens are 1-3 characters** — common words ("the", "of", "is") + leading-space variants
2. **Long-tail of "exotic" tokens** — domain jargon, numbers, code identifiers; these hit byte-fallback at sample time

The shape of this distribution is a **diagnostic** — a tokenizer trained on web text has a different distribution from one trained on code (heavy on `_`, `()`, `[]`, indentation).


In [ ]:
# Cohen proj 3 — token-length frequency table
import pandas as pd
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
text = open("pride_and_prejudice.txt").read()
ids  = tok.encode(text, add_special_tokens=False)
lens = [len(tok.decode([i]).strip()) for i in ids]      # chars per token

df = pd.Series(lens).value_counts().sort_index()
df_pct = (df / df.sum() * 100).round(1)
print(pd.DataFrame({"count": df, "pct": df_pct}))


## 4 · Characters vs bytes — the multilingual headroom

Cohen proj 4 contrasts **characters** (Unicode codepoints) with **bytes** (UTF-8). For ASCII they coincide; for everything else they diverge sharply:

| Script | char | UTF-8 bytes | Ratio |
|---|---|---|---|
| English `Hello` | 5 | 5 | 1.0 |
| French `Café` | 4 | 5 | 1.25 |
| Hindi `नमस्ते` | 6 | 18 | 3.0 |
| Tamil `வணக்கம்` | 7 | 21 | 3.0 |
| Korean `안녕하세요` | 5 | 15 | 3.0 |
| Chinese `你好` | 2 | 6 | 3.0 |
| Arabic `مرحبا` | 5 | 10 | 2.0 |
| Emoji `🦙🌟` | 2 | 8 | 4.0 |

Byte-level BPE (GPT-2, GPT-4o, Llama-3) operates on bytes — so **non-English text inflates token count by ~2-4×**. This is the **tokenization tax** of multilingual LLMs.


## 5 · Is tokenization compression?

Cohen proj 5 asks: **is BPE compressing the data?** The right metric is **bits per character (BPC)**:

```
BPC = total_bits_in_token_ids / total_chars_in_text
```

For a tokenizer with vocab `V`, each token takes `log2(V)` bits on the wire. Pride & Prejudice in:
- **Raw UTF-8**: 8 BPC
- **GPT-2 BPE (V=50,257)**: `~135k tokens × log2(50,257) / 600k chars ≈ 3.5 BPC` — **57% compression**
- **Tiktoken cl100k (V=100,277)**: `~3.4 BPC`
- **Tiktoken o200k (V=200,019)**: `~3.3 BPC`

Bigger vocab = fewer tokens per text, but bigger embedding table. The sweet spot is around 100k-200k for general-purpose models. **Llama-3 went from 32k → 128k vocab between Llama-2 and Llama-3** — one of the simplest wins of the upgrade.

> 📐 **Information-theoretic minimum.** English text has an entropy of ~1.0-1.5 BPC (Shannon 1951). Modern tokenizers reach ~3-3.5 BPC; the gap (the inefficiency) is what compression-based LMs like LZW would catch, but Transformers care about *learnable* representations, not raw compression.


## 6 · Multilingual fairness — the tokenizer-tax inequality

Cohen proj 6 generalizes: if a tokenizer was trained on a corpus that's 90% English, it spends fewer tokens per character on English than on Hindi/Tamil/Korean. **Same paragraph translated**:

| Language | Chars | GPT-4o tokens | Tokens/char |
|---|---|---|---|
| English | 100 | 21 | 0.21 |
| Spanish | 110 | 27 | 0.25 |
| Hindi   | 130 | 102 | 0.78 |
| Tamil   | 140 | 121 | 0.86 |
| Korean  | 75 | 68 | 0.91 |
| Yoruba  | 120 | 154 | 1.28 |

Implications:
- **API cost is 3-6× higher** for low-resource languages on commercial APIs
- **Context window fills 3-6× faster** — same 128k context = less content in non-English
- **In-context learning capacity** is also reduced — fewer demonstrations fit

**Llama-3.1** and **Gemini-2.5-Pro** explicitly retrained tokenizers on multilingual corpora to reduce this gap. **Cohere's Aya** and **Sea-LION** (Singapore) are open multilingual models built around this concern.

> 🌏 **Fairness side-effect (M100 callback).** Cost inequality is a *demographic-parity* violation when the API is the gatekeeper. EU AI Act + Brazil LGPD increasingly treat language-based pricing disparities as discriminatory.


## 7 · Vocabulary collisions, leading-space tokens, byte-fallback

**Leading-space tokens.** BPE distinguishes `" the"` from `"the"` — they are different token IDs (with leading space being more common in real text). Prompt-engineering pitfall: trailing spaces in your prompt change tokenization.

**Byte-fallback.** When a token isn't in the vocab (rare characters, novel emoji, non-printable bytes), modern BPEs fall back to **byte-level encoding** — each unknown byte becomes its own token. SentencePiece-Unigram instead uses an `<unk>` token unless `--byte_fallback` is enabled. Llama-3, Mistral, and DeepSeek all use byte-fallback BPE.

**Special tokens — the ChatML problem.** Modern chat templates wrap content in `<|im_start|>user`, `<|im_end|>`, `<|tool_call|>`, etc. If you tokenize raw user text these get embedded literally; if you encode through a chat template they map to single token IDs. **Always use `tokenizer.apply_chat_template(...)`**, never naive concatenation, when building a chat prompt.

> 🧨 **The "subtle token" attack.** If you append `<|im_end|>` to a user message and the model parses it as a real special token, you've **jailbroken the chat template** (M91 / M89 callback). Modern tokenizers ship with a `special_tokens` flag that, when off, prevents tokens from `<|...|>` strings being recognized as control tokens.


## 8 · Tokenizer fingerprinting

A 2024-era diagnostic: **identifying a model purely from how it splits a single string**.

```
"hello world!"  →
  GPT-2 / GPT-3.5 :  [' world', '!']
  GPT-4o / o200k  :  [' world', '!']
  Llama-3         :  [' world', '!']
  Llama-2         :  [' world', '!']
  Claude-tokenizer:  [' world', '!']
  Gemma-3         :  [' world', '!']

"def fib(n):"  →
  GPT-4o   : ['def', ' fib', '(', 'n', '):']    ← 5 tokens
  Llama-3  : ['def', ' fib', '(n', '):',]       ← 4 tokens
  Code-Llama: [code-specialized — 3 tokens]
```

Different splits arise from training-data mix (code-heavy → fewer tokens for code). Cohen's proj 7 builds a *classifier* that fingerprints a model from a known-text input + token IDs.

**Practical use:** when scraping unknown LLM API outputs (or inspecting a closed model's behaviour), the tokenizer is the most leaky surface — a few well-chosen probe strings can narrow the suspect to a small family.


## 9 · Practical pitfalls before you fine-tune

| Pitfall | Symptom | Fix |
|---|---|---|
| Mismatched tokenizer at train/serve | Garbage outputs | Always log + hash the tokenizer alongside model weights |
| Special-token leakage | Prompt injection via `<|im_end|>` | Tokenize with `add_special_tokens=False` for raw text |
| Vocab change after pretraining | Embedding-table size mismatch | Use `tok_resize_embeddings()`; retrain new rows |
| Truncation mid-utterance | Lost final assistant turn | `truncation="left"` + chat-template-aware truncation |
| Byte-fallback at inference | Slow sampling on emojis / non-Latin | Pre-warm the vocab; or fine-tune on target language |
| Leading-space inconsistency | Eval scores drop 1-3 pts vs paper | Match the paper's encoding (`add_prefix_space=True/False`) |
| BOS/EOS misuse | Model "trails off" or "doesn't start" | Check `tokenizer.bos_token_id` and pre/post-pend correctly |

> 🛠️ **Pre-finetune checklist (5 lines).** Print `tokenizer.special_tokens_map`, `tokenizer.vocab_size`, the token IDs for `<|im_start|>` / `<|im_end|>` / `<|tool|>`, the BOS/EOS pair, and the **chat template string**. Match every one against the base-model's official card before you start.


## 10 · The 2026 picker

| You're doing... | Tokenizer pick |
|---|---|
| Fine-tuning Llama-3.1 / 3.2 / 3.3 | Llama-3 tokenizer (`tiktoken-style BPE`, 128k vocab) |
| Fine-tuning Mistral / Mixtral | Mistral tokenizer (32k or 32.7k vocab) |
| Fine-tuning Qwen-2.5 / 3 | Qwen tokenizer (151k vocab, multilingual-strong) |
| Production OpenAI API | `tiktoken cl100k_base` (GPT-3.5/4) or `o200k_base` (GPT-4o, o1, o3) |
| Production Anthropic API | Claude tokenizer (closed; use Anthropic SDK `client.count_tokens(...)`) |
| Open multilingual | **Cohere Aya** (255 langs), Sea-LION, Llama-3 (128k) |
| Code-heavy | DeepSeek-Coder, CodeLlama, Qwen-Coder tokenizer |
| Edge / on-device (M90) | Phi-3.5 (32k), Gemma-3 (256k), MobileLLM (32k) |
| Math / scientific | DeepSeek-Math, Llemma — math-tuned vocab |

**The 2026 default for a *new* fine-tune of a popular base model:** stick with that base's tokenizer. Never substitute unless you're doing a vocabulary-transfer project (and even then plan ~$10k+ of compute to repair the embedding table).

> 🎓 **The Tokenization Workshop ends with one practical habit**: every time you read about a new model or evaluate one for production, look up its tokenizer's **vocab size**, **byte-level vs SentencePiece**, **multilingual coverage**, and **special-token catalogue**. That five-minute check has saved many engineering teams weeks of mystified debugging.

## ✅ Recap

- **BPE / WordPiece / Unigram + Tiktoken** are the four production families. Llama/GPT/DeepSeek/Mistral use byte-level BPE; T5/PaLM use Unigram.
- **Token-length distribution** is a tokenizer's fingerprint; ~50% are 1-3 chars.
- **Tokens vs chars vs bytes**: non-English costs 2-4× more bytes → more tokens → higher API cost + faster context fill.
- **BPC** (bits per character) measures compression; modern BPE ~3-3.5 BPC; Shannon entropy lower bound is ~1.0-1.5 BPC.
- **Multilingual fairness**: Hindi/Tamil/Korean pay 3-6× the per-character cost vs English on most APIs.
- **Pitfalls**: leading-space tokens, byte-fallback, special-token injection, embedding-table mismatch, chat-template handling.
- **2026 picker**: don't substitute a base model's tokenizer; check vocab + multilingual coverage + special tokens before any fine-tune.

Next: **M102 — Embedding Geometry Lab** (Cohen Ch3).
